# Лабораторная работа 2

Сбор книг категории Mystery с учебного сайта https://books.toscrape.com/
через BeautifulSoup. No-code часть выполняется отдельно.

## Настройка запросов

Создадим сессию с двумя повторными попытками и зададим категорию.
В каждом запросе используется timeout 5 секунд.

In [1]:
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

retry_strategy = Retry(
    total=2,
    backoff_factor=0.1,
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=['GET']
)
session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=retry_strategy))

base_url = 'https://books.toscrape.com/'
category_name = 'Mystery'

## Поиск категории

Селектор `div.side_categories ul li ul li a` находит ссылки категорий
в боковом меню. Выбираем Mystery и получаем её абсолютный URL.

In [2]:
response = session.get(base_url, timeout=5)
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')
categories_bs = {
    link.get_text(strip=True): urljoin(base_url, link['href'])
    for link in soup.select('div.side_categories ul li ul li a')
}
category_url_bs = categories_bs[category_name]
print('Категория:', category_name, category_url_bs)

Категория: Mystery https://books.toscrape.com/catalogue/category/books/mystery_3/index.html


## Сбор книг с пагинацией

`article.product_pod` — карточка книги; `h3 a` — полное название
в атрибуте `title` и ссылка; `p.price_color` — цена в фунтах.
`li.next a` — переход на следующую страницу.
Пробелы в названиях очищаем, цену приводим к float. Категория общая для всех книг.
Идентификатора в `data-*` у карточек нет, поэтому отдельное поле ID не добавляем.

In [3]:
books_bs = []
current_url = category_url_bs

while current_url:
    response = session.get(current_url, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    for product in soup.select('article.product_pod'):
        book_link = product.select_one('h3 a')
        title = ' '.join(book_link['title'].split())
        price = float(product.select_one('p.price_color').get_text(strip=True).replace('£', ''))
        link = urljoin(current_url, book_link['href'])
        books_bs.append({'title': title, 'category': category_name, 'price': price, 'link': link})

    print('Страница:', current_url)
    next_page = soup.select_one('li.next a')
    current_url = urljoin(current_url, next_page['href']) if next_page else None

session.close()

Страница: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Страница: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html


## Очистка, проверка и сохранение

Удаляем дубликаты по ссылке, выводим число строк, долю пропусков
среди всех значений и первые пять записей. Сохраняем четыре поля в `books.csv`.

In [4]:
books_bs = list({book['link']: book for book in books_bs}.values())
fields = ['title', 'category', 'price', 'link']
missing = sum(value in (None, '') for book in books_bs for value in book.values())
print('Число строк:', len(books_bs))
print(f'Доля пропусков: {missing / (len(books_bs) * len(fields)):.2%}')
for book in books_bs[:5]:
    print(book)

with open('books.csv', 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(books_bs)

print('Данные сохранены в books.csv')

Число строк: 32
Доля пропусков: 0.00%
{'title': 'Sharp Objects', 'category': 'Mystery', 'price': 47.82, 'link': 'https://books.toscrape.com/catalogue/sharp-objects_997/index.html'}
{'title': 'In a Dark, Dark Wood', 'category': 'Mystery', 'price': 19.63, 'link': 'https://books.toscrape.com/catalogue/in-a-dark-dark-wood_963/index.html'}
{'title': 'The Past Never Ends', 'category': 'Mystery', 'price': 56.5, 'link': 'https://books.toscrape.com/catalogue/the-past-never-ends_942/index.html'}
{'title': 'A Murder in Time', 'category': 'Mystery', 'price': 16.64, 'link': 'https://books.toscrape.com/catalogue/a-murder-in-time_877/index.html'}
{'title': 'The Murder of Roger Ackroyd (Hercule Poirot #4)', 'category': 'Mystery', 'price': 44.1, 'link': 'https://books.toscrape.com/catalogue/the-murder-of-roger-ackroyd-hercule-poirot-4_852/index.html'}
Данные сохранены в books.csv
